In [ ]:
%pip install -q -U git+https://github.com/huggingface/transformers accelerate pillow qwen-vl-utils "datasets==3.6.0"


In [ ]:
from pathlib import Path
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
QUESTIONS = {
    42: "What color are the gym shoes?",
    73: "Is this a motorcycle or bike?",
    74: "Does this dog have a collar?",
    133: "What color is lamp?",
    136: "Is this in a museum?",
    139: "What is the woman in the room doing?",
    143: "How many birds are in the tree?",
    164: "What is the color of the refridgerator?",
    192: "What sport is being played?",
    196: "What is the yellow food?",
    208: "What material is the countertop made of?",
    241: "What is he sitting on?",
    257: "Is the dog real?",
    283: "What is the brand of wine?",
    285: "Is it daytime?",
    294: "Are these people having fun?",
    328: "What are the men sitting on?",
    338: "Is this a bedroom?",
    357: "Who are the men in black?",
    359: "Overcast or sunny?",
    360: "Are the streetlamps on?",
}

image_paths = sorted(Path(".").glob("*.jpg"))
samples = [(int(p.stem.rsplit("_", 1)[-1]), p, QUESTIONS[int(p.stem.rsplit("_", 1)[-1])]) for p in image_paths]


In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda", attn_implementation="eager"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)


In [ ]:
def prepare_inputs(image_path, question):
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    return {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

def vision_embeds(inputs):
    pixel_values = inputs["pixel_values"].type(model.model.visual.get_dtype() if hasattr(model.model.visual, "get_dtype") else inputs["pixel_values"].dtype)
    with torch.no_grad():
        embeds = model.model.visual(pixel_values, grid_thw=inputs["image_grid_thw"])
    return embeds.pooler_output if hasattr(embeds, "pooler_output") else embeds

def build_pruned_inputs(inputs, vision_importance, keep_ratio=0.5):
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    image_embeds = vision_embeds(inputs)
    image_positions = torch.where(input_ids[0] == model.config.image_token_id)[0]
    if image_embeds.shape[0] != image_positions.numel():
        raise ValueError(f"image_embeds={image_embeds.shape[0]} image_positions={image_positions.numel()}")
    k = max(1, round(image_embeds.shape[0] * keep_ratio))
    keep_idx = torch.sort(torch.topk(vision_importance.to(model.device), k).indices).values
    keep_seq = torch.ones_like(input_ids[0], dtype=torch.bool, device=model.device)
    keep_seq[image_positions] = False
    keep_seq[image_positions[keep_idx]] = True
    text_embeds = model.model.language_model.embed_tokens(input_ids)
    new_ids = input_ids[:, keep_seq]
    new_embeds = text_embeds[:, keep_seq].clone()
    new_embeds[0, new_ids[0] == model.config.image_token_id] = image_embeds[keep_idx].to(new_embeds.dtype)
    position_ids, rope_deltas = model.model.get_rope_index(
        input_ids=input_ids,
        image_grid_thw=inputs["image_grid_thw"],
        attention_mask=attention_mask,
        mm_token_type_ids=inputs["mm_token_type_ids"],
    )
    return {
        "input_ids": new_ids,
        "inputs_embeds": new_embeds,
        "attention_mask": attention_mask[:, keep_seq],
        "mm_token_type_ids": inputs["mm_token_type_ids"][:, keep_seq],
        "position_ids": position_ids[:, :, keep_seq],
        "rope_deltas": rope_deltas,
        "keep_idx": keep_idx,
    }

def run_pruned(inputs, vision_importance, keep_ratio=0.5, max_new_tokens=32):
    pruned = build_pruned_inputs(inputs, vision_importance, keep_ratio)
    with torch.no_grad():
        ids = model.generate(
            inputs_embeds=pruned["inputs_embeds"],
            attention_mask=pruned["attention_mask"],
            position_ids=pruned["position_ids"],
            rope_deltas=pruned["rope_deltas"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    return ids, processor.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0], pruned


In [ ]:
sample_idx = 10
sample_id, image_path, question = samples[sample_idx]
importance_path = Path(f"importances_test/debiased_importance_{sample_id}.pt")
importance = torch.load(importance_path, map_location="cpu")[10].mean(0)
inputs = prepare_inputs(image_path, question)
generated_ids, answer, pruned = run_pruned(inputs, importance, keep_ratio=0.5)
print(sample_id, question, answer, pruned["input_ids"].shape)
